# GPT-OSS Reference Profiling Notebook

Profile GPT-OSS with dummy BF16 weights and the forward path in `model.py`.

## How To Use

1. Edit the next code cell.
2. Every architecture field from `ModelConfig` is listed there and can be changed directly.
3. Run the preflight cell before running the model.
4. Start with the smallest workload: `prefill_tokens=1`, `generated_tokens=1`, `warmup_iters=0`, `measure_iters=1`.
5. Only enable the sweep cell after a safe single run.

## Decode Semantics

Strict-reference mode uses one prompt forward pass for prefill. If `generated_tokens > 0`, the first new token is sampled from the prefill logits. `decode_total` measures only the follow-on full-sequence passes for the remaining generated tokens.

## Note On Memory

This reference MoE path materializes large temporary expert tensors. On unified-memory systems such as DGX Spark, oversized runs can make the machine unresponsive instead of raising a clean CUDA OOM. Use the preflight output as a guard and scale up gradually.


In [ ]:
from dataclasses import asdict
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

CWD = Path.cwd().resolve()
REPO_ROOT = next(
    (path for path in (CWD, *CWD.parents) if (path / "pyproject.toml").exists()),
    CWD,
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from profiling import (
    DEFAULT_GENERATED_SWEEP,
    DEFAULT_PREFILL_SWEEP,
    TimingConfig,
    WorkloadConfig,
    format_bytes,
    model_config_from_preset,
    plot_level0_breakdown,
    plot_level1_breakdown,
    plot_level2_heatmap,
    plot_workload_grid,
    preflight_report,
    run_workload,
    run_workload_sweep,
    summarize_results,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


In [ ]:
PRESET_NAME = "gpt-oss-20b"  # "gpt-oss-20b", "gpt-oss-120b", or "custom"

_BASE_ARCHITECTURE = asdict(model_config_from_preset(PRESET_NAME))
ARCHITECTURE = {
    "num_hidden_layers": _BASE_ARCHITECTURE["num_hidden_layers"],
    "num_experts": _BASE_ARCHITECTURE["num_experts"],
    "experts_per_token": _BASE_ARCHITECTURE["experts_per_token"],
    "vocab_size": _BASE_ARCHITECTURE["vocab_size"],
    "hidden_size": _BASE_ARCHITECTURE["hidden_size"],
    "intermediate_size": _BASE_ARCHITECTURE["intermediate_size"],
    "swiglu_limit": _BASE_ARCHITECTURE["swiglu_limit"],
    "head_dim": _BASE_ARCHITECTURE["head_dim"],
    "num_attention_heads": _BASE_ARCHITECTURE["num_attention_heads"],
    "num_key_value_heads": _BASE_ARCHITECTURE["num_key_value_heads"],
    "sliding_window": _BASE_ARCHITECTURE["sliding_window"],
    "initial_context_length": _BASE_ARCHITECTURE["initial_context_length"],
    "rope_theta": _BASE_ARCHITECTURE["rope_theta"],
    "rope_scaling_factor": _BASE_ARCHITECTURE["rope_scaling_factor"],
    "rope_ntk_alpha": _BASE_ARCHITECTURE["rope_ntk_alpha"],
    "rope_ntk_beta": _BASE_ARCHITECTURE["rope_ntk_beta"],
}

DEVICE = "cuda"
SEED = 0

SINGLE_WORKLOAD = {
    "prefill_tokens": 1,
    "generated_tokens": 1,
}

RUN_SWEEP = False
SWEEP_PREFILL = list(DEFAULT_PREFILL_SWEEP)
SWEEP_GENERATED = list(DEFAULT_GENERATED_SWEEP)

TIMING = TimingConfig(
    warmup_iters=0,
    measure_iters=1,
    seed=SEED,
    enable_torch_profiler=False,
    trace_output_dir="traces",
)


In [ ]:
arch_cfg = model_config_from_preset(PRESET_NAME, ARCHITECTURE)
single_workload = WorkloadConfig(**SINGLE_WORKLOAD)
report = preflight_report(arch_cfg, single_workload, device=DEVICE)

arch_table = pd.DataFrame(
    [{"field": key, "value": value} for key, value in asdict(arch_cfg).items()]
)

preflight_table = pd.DataFrame(
    [{"metric": key, "value": value} for key, value in report.items()]
)
preflight_table["pretty_value"] = preflight_table.apply(
    lambda row: format_bytes(row["value"])
    if isinstance(row["value"], int) and ("bytes" in row["metric"])
    else row["value"],
    axis=1,
)

important_metrics = [
    "device_total_bytes",
    "device_free_bytes",
    "dense_weight_bytes",
    "peak_runtime_component",
    "peak_runtime_bytes",
    "moe_mlp1_temp_bytes",
    "moe_mlp2_temp_bytes",
    "logits_bytes",
    "estimated_peak_bytes",
    "fits_free_memory",
]

display(arch_table)
display(
    preflight_table[preflight_table["metric"].isin(important_metrics)][
        ["metric", "pretty_value"]
    ]
)

if report.get("fits_free_memory") is False:
    print("Preflight says this run is unsafe. Reduce model size or token counts before running the model.")
else:
    print("Preflight passed for the current workload.")


In [ ]:
if DEVICE == "cuda" and report.get("fits_free_memory") is False:
    raise RuntimeError(
        "Preflight failed. Reduce model size or token counts before running the model."
    )

single_rows = run_workload(
    arch_cfg,
    single_workload,
    TIMING,
    device=DEVICE,
    preset_name=PRESET_NAME,
)
single_summary = summarize_results(single_rows)

trace_path = single_rows.attrs.get("trace_path")
if trace_path:
    print(f"Chrome trace written to: {trace_path}")


In [ ]:
display(single_summary["level0_metrics"])
display(single_summary["level1"].sort_values(["phase", "scope_name"]))
display(
    single_summary["level2"]
    .sort_values(["phase", "component", "layer_idx", "scope_name"])
    .reset_index(drop=True)
)


In [ ]:
plot_level0_breakdown(single_summary["level0_metrics"])
plot_level1_breakdown(single_summary["level1"], phase="combined")
plot_level2_heatmap(
    single_summary["level2"],
    phase="prefill",
    component="attention",
    prefill_tokens=SINGLE_WORKLOAD["prefill_tokens"],
    generated_tokens=SINGLE_WORKLOAD["generated_tokens"],
)
plot_level2_heatmap(
    single_summary["level2"],
    phase="prefill",
    component="mlp",
    prefill_tokens=SINGLE_WORKLOAD["prefill_tokens"],
    generated_tokens=SINGLE_WORKLOAD["generated_tokens"],
)


In [ ]:
if not RUN_SWEEP:
    print("Set RUN_SWEEP = True after a safe single run to execute the sweep.")
else:
    sweep_rows = run_workload_sweep(
        arch_cfg,
        prefill_tokens=SWEEP_PREFILL,
        generated_tokens=SWEEP_GENERATED,
        timing_cfg=TIMING,
        device=DEVICE,
        preset_name=PRESET_NAME,
    )
    sweep_summary = summarize_results(sweep_rows)

    display(
        sweep_summary["level0_metrics"]
        .sort_values(["prefill_tokens", "generated_tokens"])
        .reset_index(drop=True)
    )

    plot_workload_grid(sweep_summary["level0_metrics"], value="workload_total")
    plot_workload_grid(sweep_summary["level0_metrics"], value="avg_ms_per_new_token")
    plot_level1_breakdown(sweep_summary["level1"], phase="combined")
